In [14]:
import torch

torch.manual_seed(42)

In [15]:
features = torch.tensor([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

true_weights = torch.tensor([
    [2.0],
    [-3.0],
])

true_bias = torch.tensor([5.0])

# [4,2] @ [2,1] + (1,) -> (4,1)
labels = features @ true_weights + true_bias

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)
print("Labels:")
print(labels)

Features shape: torch.Size([4, 2])
Labels shape: torch.Size([4, 1])
Labels:
tensor([[ 1.],
        [ 6.],
        [-1.],
        [ 4.]])


In [16]:
# 학습 전 parameter를 임의의 값으로 초기화
weights = torch.tensor(
    [[0.5], [-0.5]],
    requires_grad=True,
)

bias = torch.tensor(
    [0.0],
    requires_grad=True,
)

# 선형회귀의 forward 계산: ŷ=Xw+b
predictions = features @ weights + bias

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)
print("Predictions shape:", predictions.shape)
print("Predictions:")
print(predictions)

Weights shape: torch.Size([2, 1])
Bias shape: torch.Size([1])
Predictions shape: torch.Size([4, 1])
Predictions:
tensor([[-0.5000],
        [ 0.5000],
        [-0.5000],
        [ 0.5000]], grad_fn=<AddBackward0>)


In [17]:
# 각 example의 residual과 squared loss를 계산
residuals = predictions - labels

# 1/2를 곱하면 backward에서 제곱의 2가 상쇄
per_example_loss = 0.5 * residuals.pow(2)

# 전체 training loss는 example별 loss의 평균
loss = per_example_loss.mean()

print("Residuals:")
print(residuals)

print("\nPer-example loss:")
print(per_example_loss)

print("\nMean loss:", loss)

Residuals:
tensor([[-1.5000],
        [-5.5000],
        [ 0.5000],
        [-3.5000]], grad_fn=<SubBackward0>)

Per-example loss:
tensor([[ 1.1250],
        [15.1250],
        [ 0.1250],
        [ 6.1250]], grad_fn=<MulBackward0>)

Mean loss: tensor(5.6250, grad_fn=<MeanBackward0>)


In [18]:
# scalar loss에서 weight와 bias까지 gradient를 계산
loss.backward()

print("Gradient of weights:")
print(weights.grad)

print("\nGradient of bias:")
print(bias.grad)

Gradient of weights:
tensor([[-6.2500],
        [-4.2500]])

Gradient of bias:
tensor([-2.5000])


In [ ]:
# minibatch SGD update를 한 번 직접 수행한다.
learning_rate = 0.1

with torch.no_grad():
    # negative gradient 방향으로 parameter를 이동한다.
    weights -= learning_rate * weights.grad
    bias -= learning_rate * bias.grad

    # 다음 backward 전에 gradient buffer를 반드시 초기화한다.
    weights.grad.zero_()
    bias.grad.zero_()

updated_predictions = features @ weights + bias
updated_loss = (
    0.5
    * (updated_predictions - labels)**2
).mean()

print("Loss before update:", loss.item())
print("Loss after update:", updated_loss.item())

assert updated_loss < loss

Loss before update: 5.625
Loss after update: 3.976562261581421


In [22]:
# Bias를 feature 행렬의 마지막 열인 1과 합쳐 normal equation으로 풀어보기
from cProfile import label


ones = torch.ones(
    (features.shape[0], 1),
)

augmented_features = torch.cat(
    (features, ones),
    dim=1,
)

# 역행렬을 직접 구하지 않고 선형 연립방정식을 푼다.
analytic_parameters = torch.linalg.solve(
    augmented_features.T @ augmented_features,
    augmented_features.T @ labels,
)

analytic_weights = analytic_parameters[:2]
analytic_bias = analytic_parameters[2:]

print("Analytic weights:")
print(analytic_weights)

print("\nAnalytic bias:")
print(analytic_bias)

Analytic weights:
tensor([[ 2.0000],
        [-3.0000]])

Analytic bias:
tensor([[5.]])
